# 1. Filter Massive OSM Dataset


In [1]:
from pathlib import Path

import geopandas as gpd
from shapely.geometry import Polygon


def filter_lta_layer_by_polygon(shapefile_path, output_path, polygon_coords):
    print(f"🔄 Processing {shapefile_path}...")

    # 1. Read the LTA Shapefile
    gdf = gpd.read_file(shapefile_path)

    # 2. Match Singapore's SVY21 coordinate system (EPSG:3414)
    # to standard GPS coordinates (WGS84 - EPSG:4326) if necessary
    if gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    # 3. Create a Shapely Polygon from the coordinate list
    # Crucial: Shapely expects (Longitude, Latitude) order
    poly_points = [(pt["lng"], pt["lat"]) for pt in polygon_coords]
    neighborhood_polygon = Polygon(poly_points)

    # 4. Filter data intersecting this polygon (Keeps intersecting features intact)
    filtered_gdf = gdf[gdf.geometry.intersects(neighborhood_polygon)]

    # 5. Export back as a clean, localized Shapefile bundle
    filtered_gdf.to_file(output_path)
    print(
        f"✅ Saved localized layer to {output_path} ({len(filtered_gdf)} features found)"
    )


# Get list of shapefiles in raw_data folder
# Loop all folders in GEOSPATIAL folder, get only the string right before underscore to be the shapefile names
OSM_DIR = Path("malaysia-singapore-brunei-260706-free.shp")
OSM_CLEMENTI_MALL_DIR = Path("OSM_CLEMENTI_MALL")
# Loop subfolders only (skip .zip files)
shapefile_names = []

# --- CONFIGURATION FOR YOUR HDB ESTATE ---
clementi_poly_coords = [
    {"lat": 1.3161273531516402, "lng": 103.76451730728151},
    {"lat": 1.3148616841675702, "lng": 103.76318693161012},
    {"lat": 1.3131026177342597, "lng": 103.76329421997072},
    {"lat": 1.3111719336386278, "lng": 103.76516103744507},
    {"lat": 1.3126092208293652, "lng": 103.7667489051819},
]

shp_files = list(OSM_DIR.glob("*.shp"))
for layer in shp_files:
    shapefile_names.append(layer.stem)

# Run the filter for your downloaded layers (Update names according to your extracted file filenames)
for filename in shapefile_names:
    filter_lta_layer_by_polygon(
        str(OSM_DIR / f"{filename}.shp"),
        str(OSM_CLEMENTI_MALL_DIR / f"{filename}.shp"),
        clementi_poly_coords,
    )

# Run the filter for your downloaded layers (Update names according to your extracted file filenames)
# filter_lta_layer("raw_data/Footpath.shp", "clementi_footpaths.shp", CLEMENTI_BBOX)
# filter_lta_layer("raw_data/KerbLine.shp", "clementi_kerblines.shp", CLEMENTI_BBOX)

🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_landuse_a_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI_MALL/gis_osm_landuse_a_free_1.shp (16 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_natural_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI_MALL/gis_osm_natural_free_1.shp (0 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_buildings_a_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI_MALL/gis_osm_buildings_a_free_1.shp (52 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_waterways_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI_MALL/gis_osm_waterways_free_1.shp (0 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_pofw_free_1.shp...
✅ Saved localized layer to OSM_CLEMENTI_MALL/gis_osm_pofw_free_1.shp (0 features found)
🔄 Processing malaysia-singapore-brunei-260706-free.shp/gis_osm_water_a_free_1.shp...
✅ Saved localized layer

In [ ]:
import json
import geopandas as gpd

# Initialize your final frontend data payload
threejs_payload = {"paths": [], "points": []}

In [3]:
# --- 1. PROCESS ROADS LINES ---
# Read your cropped footpath shapefile
roads = gpd.read_file("./OSM_CLEMENTI_MALL/gis_osm_roads_free_1.shp")

print(roads.head())
print(roads.columns)

      osm_id  code       fclass                      name   ref oneway  \
0   22892482  5122  residential         Clementi Avenue 3  None      F   
1   75009416  5113      primary  Commonwealth Avenue West  None      F   
2  105823594  5141      service                       NaN  None      B   
3  119879864  5141      service                       NaN  None      F   
4  155926483  5122  residential         Clementi Avenue 3  None      F   

   maxspeed  layer bridge tunnel  \
0        40      0      F      F   
1        60      0      F      F   
2         0      0      F      F   
3        20      0      F      F   
4        40      0      F      F   

                                            geometry  
0  LINESTRING (103.76346 1.31459, 103.76347 1.314...  
1  LINESTRING (103.76672 1.31258, 103.76656 1.312...  
2  LINESTRING (103.76345 1.31434, 103.76339 1.314...  
3  LINESTRING (103.7651 1.31227, 103.76507 1.3123...  
4  LINESTRING (103.7651 1.31154, 103.76505 1.3115...  
Index(['

In [ ]:
# get unique values of the column "fclass"
print(roads["fclass"].unique())
print(roads["name"].unique())
print(roads["oneway"].unique())

<StringArray>
[ 'residential',      'primary',      'service', 'primary_link',
      'footway',     'cycleway',        'steps']
Length: 7, dtype: str
<StringArray>
[                                                'Clementi Avenue 3',
                                          'Commonwealth Avenue West',
                                                                 nan,
                                          'Clementi Bus Interchange',
 'Bridge inside Clementi MRT Station that connects to Clementi Mall',
                                'Bridge inside Clementi MRT Station']
Length: 6, dtype: str
<StringArray>
['F', 'B']
Length: 2, dtype: str


In [7]:
import geopandas as gpd
import json

# Load your cropped OSM roads shapefile
roads_gdf = gpd.read_file("./OSM_CLEMENTI_MALL/gis_osm_roads_free_1.shp")

processed_3d_roads = []

# Reference map anchor center (using first coordinate to prevent 3D floating point jitter)
first_geom = roads_gdf.geometry.iloc[0]
if first_geom.geom_type == "LineString":
    center_lng, center_lat = first_geom.coords[0]
else:
    center_lng, center_lat = list(first_geom.geoms[0].coords)[0]

scale_factor = 100000  # Scale degree coordinates into local viewport units

for idx, row in roads_gdf.iterrows():
    geom = row["geometry"]
    if geom.is_empty:
        continue

    # Extract fclass string safely
    fclass = str(row["fclass"])

    # 1. ASSIGN PHYSICAL MESH WIDTHS
    if fclass == "primary":
        width = 12.0
        color = "#222222"
    elif fclass == "primary_link":
        width = 8.0
        color = "#2b2b2b"
    elif fclass == "residential":
        width = 6.0
        color = "#333333"
    elif fclass == "service":
        width = 4.0
        color = "#444444"
    elif fclass in ["footway", "pedestrian"]:
        width = 2.5
        color = "#33ff33"  # Green for walking channels
    elif fclass == "cycleway":
        width = 2.0
        color = "#00ffff"
    elif fclass == "steps":
        width = 2.0
        color = "#aaaaaa"
    else:
        width = 3.5  # Fallback standard single lane width
        color = "#555555"

    # 2. CALCULATE 3D VERTICAL STACKING (ELEVATION & THICKNESS)
    # Cast layer attribute to integer safely (default to 0 if null)
    layer_val = int(row["layer"]) if "layer" in row and row["layer"] is not None else 0
    base_y_elevation = layer_val * 4.0  # 4 meters per structural floor tier

    is_bridge = str(row["bridge"]) == "T" if "bridge" in row else False
    is_tunnel = str(row["tunnel"]) == "T" if "tunnel" in row else False

    mesh_thickness = 0.2  # Default thin flat road sheet
    if is_bridge:
        mesh_thickness = 1.5  # Heavy block look for overhead flyovers
    elif is_tunnel:
        base_y_elevation -= 1.0  # Push slightly below the terrain mesh surface

    # 3. COORDINATE TRANSFORMATION LOOP
    # Handle both standard LineStrings and rare MultiLineStrings cleanly
    linestrings = [geom] if geom.geom_type == "LineString" else list(geom.geoms)

    for line in linestrings:
        # Convert absolute GPS coords into scaled local offsets relative to center anchor
        local_coordinates = []
        for lng, lat in line.coords:
            x = (lng - center_lng) * scale_factor
            z = (
                -(lat - center_lat) * scale_factor
            )  # Invert latitude to map over 3D Z-axis
            local_coordinates.append([x, base_y_elevation, z])

        # 4. PACKAGE STRUCTURAL ATTRIBUTES
        processed_3d_roads.append(
            {
                "id": row.get("osm_id", idx),
                "name": row.get("name", "Unnamed Road"),
                "fclass": fclass,
                "width": width,
                "thickness": mesh_thickness,
                "elevation": base_y_elevation,
                "color_hex": color,
                "is_bridge": is_bridge,
                "is_tunnel": is_tunnel,
                "direction_mode": str(
                    row.get("oneway", "B")
                ),  # Safe lane driving flows
                "coordinates": local_coordinates,
            }
        )

# Export a dedicated clean road schema for the frontend mesh generation engine
output_payload = {
    "anchor_center": [center_lng, center_lat],
    "mesh_roads": processed_3d_roads,
}

with open("threejs_3d_roads.json", "w") as f:
    json.dump(output_payload, f, indent=4)

print(
    f"✅ Successfully converted {len(processed_3d_roads)} road vector outlines into physical 3D mesh properties!"
)

✅ Successfully converted 321 road vector outlines into physical 3D mesh properties!


# Buildings


In [8]:
# Read your cropped footpath shapefile
buildings_gdf = gpd.read_file("./OSM_CLEMENTI_MALL/gis_osm_buildings_a_free_1.shp")

print(buildings_gdf.head())
print(buildings_gdf.columns)

      osm_id  code    fclass                      name           type  \
0  104147233  1500  building                       NaN  train_station   
1  118994815  1500  building         The Clementi Mall            NaN   
2  119879838  1500  building                       NaN    residential   
3  119879841  1500  building                       NaN            NaN   
4  119879842  1500  building  Grantral Mall @ Clementi            NaN   

                                            geometry  
0  POLYGON ((103.76488 1.31564, 103.76506 1.31572...  
1  POLYGON ((103.7636 1.31445, 103.7636 1.31447, ...  
2  POLYGON ((103.764 1.31292, 103.76416 1.31298, ...  
3  POLYGON ((103.76424 1.31342, 103.76424 1.31344...  
4  POLYGON ((103.76492 1.31444, 103.76503 1.31449...  
Index(['osm_id', 'code', 'fclass', 'name', 'type', 'geometry'], dtype='str')


In [11]:
print(buildings_gdf["fclass"].unique())
print(buildings_gdf["name"].unique())
# print(buildings_gdf["geometry"].unique())
print(buildings_gdf["type"].unique())

<StringArray>
['building']
Length: 1, dtype: str
<StringArray>
[nan, 'The Clementi Mall', 'Grantral Mall @ Clementi', 'Clementi 321']
Length: 4, dtype: str
<StringArray>
['train_station',             nan,   'residential',    'commercial',
    'government',        'garage',          'roof',       'service']
Length: 8, dtype: str
